# Analyse Exploratoire des Données — Agriculture CropYield Dataset

**Objectif** : Comprendre le fichier `crop_yield.csv` avant tout nettoyage ou modélisation :
quelles colonnes, quels types, quelle qualité, et quelles variables semblent liées au rendement.

**Dataset** :  `data/agriculture-crop-yield/crop_yield.csv`
 - Une ligne = **une parcelle sur une saison** : ses conditions (région, sol, culture semée, pluie,
température, engrais, irrigation, météo, durée avant récolte) et le rendement obtenu.
 - 10 colonnes :

| Colonne | Type | Ce que c'est |
|---|---|---|
| `Region` | texte | Zone géographique : North, East, South, West |
| `Soil_Type` | texte | Type de sol : Sandy, Clay, Loam, Silt, Peaty, Chalky |
| `Crop` | texte | Culture semée : Wheat, Rice, Maize, Barley, Soybean, Cotton |
| `Rainfall_mm` | décimal | Pluie tombée sur la saison, en millimètres |
| `Temperature_Celsius` | décimal | Température moyenne, en degrés Celsius |
| `Fertilizer_Used` | booléen | Engrais utilisé ou non |
| `Irrigation_Used` | booléen | Irrigation utilisée ou non |
| `Weather_Condition` | texte | Météo dominante : Sunny, Rainy, Cloudy |
| `Days_to_Harvest` | entier | Nombre de jours avant la récolte |
| `Yield_tons_per_hectare` | décimal | **Cible** : rendement en tonnes par hectare |


Exemple, la première ligne du fichier :
> Une parcelle de l'**Ouest**, sol **sableux**, semée en **coton**. Elle a reçu **897 mm** de pluie,
> il a fait **27,7 °C** en moyenne, **sans engrais** mais **avec irrigation**, temps **nuageux**,
> et **122 jours** avant la récolte. Résultat : **6,56 tonnes par hectare**.

# Imports

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from data_profiling import ProfileReport

from agritech import utils
from agritech.config import AGRICULTURE_CROP_YIELD_FILENAME, PATHS


# Lecture du dataset

In [2]:
csv_path = PATHS.data_agriculture_crop_yield / AGRICULTURE_CROP_YIELD_FILENAME

In [3]:
df = pd.read_csv(csv_path)
df

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251
...,...,...,...,...,...,...,...,...,...,...
999995,West,Silt,Rice,302.805345,27.987428,False,False,Sunny,76,1.347586
999996,South,Chalky,Barley,932.991383,39.661039,True,False,Rainy,93,7.311594
999997,North,Peaty,Cotton,867.362046,24.370042,True,False,Cloudy,108,5.763182
999998,West,Silt,Wheat,492.812857,33.045505,False,False,Sunny,102,2.070159


# Statistiques basiques

In [ ]:
profile = ProfileReport(
    df,
    title="Profiling Report",
    minimal=True,
    correlations={"auto": {"calculate": True, "threshold": 0.5}},
    duplicates={"head": 5},
)
profile.to_file(PATHS.docs / "eda_crop_yield_rapport.html")

In [ ]:
desc = profile.get_description()
print(f"Observations       : {desc.table['n']:,}")
print(f"Variables          : {desc.table['n_var']}")
print(f"Cellules manquantes: {desc.table['n_cells_missing']} ({desc.table['p_cells_missing']:.2%})")
print(f"Lignes dupliquees  : {desc.table['n_duplicates']} ({desc.table['p_duplicates']:.2%})")
print(f"Taille en memoire  : {desc.table['memory_size'] / 1024**2:.1f} Mo")
print(f"Types              : {desc.table['types']}")
print("\nAlertes :")
for a in desc.alerts:
    print(" -", a)


In [ ]:
pd.DataFrame(desc.variables).T[["type", "n_distinct", "p_missing", "mean", "std", "min", "max"]]

**Observations :**

- **1 000 000 de lignes, 10 colonnes**, aucune valeur manquante, aucun doublon : rien à remplir ni à dédoublonner.
- Types correctement lus par pandas (4 textes, 4 numériques, 2 booléens), aucune conversion nécessaire.
- Les alertes « valeurs uniques » sont normales sur un million de nombres à virgule ; celle sur la corrélation pluie / rendement est en revanche un vrai signal.

# Variables Numériques

In [ ]:
num_cols = ["Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest", "Yield_tons_per_hectare"]

utils.plot_distribs(df, num_cols)

**Observations :**

- `Rainfall_mm`, `Temperature_Celsius` et `Days_to_Harvest` sont **uniformes** ; la cible suit une **courbe en cloche** centrée sur 4,65 t/ha.
- Attention à l'échelle : l'axe vertical des trois premiers graphes ne part pas de zéro, les creux visibles ne valent que **2 %** d'écart.
- Aucune distribution n'est déformée d'un côté : pas besoin de transformation avant de modéliser.

## Outliers

In [ ]:
# Règle classique : est atypique ce qui sort de [Q1 - 1,5 x IQR ; Q3 + 1,5 x IQR]
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    basse, haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < basse) | (df[col] > haute)).sum()
    print(f"{col:24} {n:6d} atypiques ({n / len(df):.3%})   bornes = [{basse:.2f}, {haute:.2f}]")

**Observations :**

- Les trois variables uniformes n'ont **aucune** valeur atypique : elles sont bornées.
- La cible compte **84** valeurs atypiques selon l'IQR (0,008 %) : 56 sous −0,27 et 28 au-dessus de 9,57. Aucune suppression automatique ; les valeurs négatives sont examinées juste après.

### Target

In [ ]:
df['Yield_tons_per_hectare'].describe()

Target négative ?
- Pas de fertilisation : 100%
- Pas d'irrigation : 100%

In [ ]:
negative_target = df[df['Yield_tons_per_hectare'] < 0]
print("Nombre de rendements négatifs : ", len(negative_target))
print("   -> Fertilizer_Used ? ", len(negative_target[negative_target['Fertilizer_Used'] == True]))
print("   -> Irrigation_Used ? ", len(negative_target[negative_target['Irrigation_Used'] == True]))

print("   -> Pluie ? ")
print("      - < 200 : ",len(negative_target[negative_target['Rainfall_mm'] < 200]))
print("      - > 200 : ",len(negative_target[negative_target['Rainfall_mm'] > 200]))
print(f"     - (min,max) : ({min(negative_target['Rainfall_mm']):3.2f},{max(negative_target['Rainfall_mm']):3.2f})")

print("   -> Température ? ")
print("      - < 20 : ",len(negative_target[negative_target['Temperature_Celsius'] < 20]))
print("      - > 20 : ",len(negative_target[negative_target['Temperature_Celsius'] > 20]))
print(f"     - (min,max) : ({min(negative_target['Temperature_Celsius']):3.2f},{max(negative_target['Temperature_Celsius']):3.2f})")
negative_target

In [ ]:
negative_target[negative_target['Rainfall_mm'] > 200]

**Observations :**

- La cible va de **−1,15** à **9,96**, moyenne et médiane à 4,65.
- **231 lignes** ont un rendement négatif (0,023 %), **toutes sans engrais ni irrigation**, et 223 sur 231 avec moins de 200 mm de pluie.
- Trois options à départager au nettoyage, aucune tranchée ici : conserver, ramener à 0, ou supprimer les lignes.

### Corrélations

In [ ]:
# Matrice pearson
utils.plot_pearson_matrix(df, target="Yield_tons_per_hectare")

In [ ]:
# Croisements avec la cible pour les variables les plus corrélées
utils.plot_pairs(
    df,
    "Yield_tons_per_hectare",
    ["Rainfall_mm", "Fertilizer_Used", "Irrigation_Used", "Temperature_Celsius", "Days_to_Harvest"],
)

**Observations :**

- Corrélation avec le rendement : `Rainfall_mm` **0,76**, `Fertilizer_Used` **0,44**, `Irrigation_Used` **0,35**, `Temperature_Celsius` **0,09**, `Days_to_Harvest` **0,00**. Entre variables d'entrée, tout est à 0,00 : aucune redondance.
- Avec engrais **+1,50 t/ha**, avec irrigation **+1,20 t/ha** ; la relation pluie / rendement est droite et de largeur constante.
- Deux vérifications suivent : la température (0,09 ne suffit pas à écarter une variable) et `Days_to_Harvest` (une corrélation nulle n'exclut qu'une relation linéaire).

### Temperature

In [ ]:
# La corrélation est faible (0,09) : cache-t-elle malgré tout une tendance régulière ?
utils.use_inline_backend()

tranches_temp = pd.cut(df["Temperature_Celsius"], bins=10)
moyennes_temp = df.groupby(tranches_temp, observed=True)["Yield_tons_per_hectare"].mean()

ax = moyennes_temp.plot(marker="o", figsize=(10, 4))
ax.set_xlabel("Tranche de température (°C)")
ax.set_ylabel("Rendement moyen (t/ha)")
ax.set_title("Rendement moyen par tranche de Temperature_Celsius")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

ecart = moyennes_temp.iloc[-1] - moyennes_temp.iloc[0]
print(f"écart entre la tranche la plus froide et la plus chaude : {ecart:+.3f} t/ha")

**Observations :**

- Le rendement moyen augmente de **4,423 à 4,867 t/ha**, soit **+0,445 t/ha** entre les tranches les plus froides et les plus chaudes, sans exception dans la progression.
- L'effet est faible par rapport aux variables dominantes, mais il est régulier dans ce jeu de données.
- Une corrélation faible (**0,09**) ne signifie donc pas que la température est inutile : elle décrit un effet faible face à la variabilité totale, que le découpage en tranches rend simplement visible.

### Days_to_Harvest

In [ ]:
# La corrélation ne voit que les relations droites.
# On regarde le rendement moyen par tranche, pour repérer une éventuelle courbe.
utils.use_inline_backend()

tranches = pd.cut(df["Days_to_Harvest"], bins=10)
moyennes = df.groupby(tranches, observed=True)["Yield_tons_per_hectare"].mean()

ax = moyennes.plot(marker="o", figsize=(10, 4))
ax.set_xlabel("Tranche de jours avant récolte")
ax.set_ylabel("Rendement moyen (t/ha)")
ax.set_title("Rendement moyen par tranche de Days_to_Harvest")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"amplitude entre tranches : {moyennes.max() - moyennes.min():.4f} t/ha")

**Observations :**

- Le rendement moyen va de 4,633 à 4,662 selon les tranches, soit **0,029 t/ha** d'amplitude, sans direction ni forme régulière.
- Il n'y a donc **pas non plus de relation en courbe** : l'absence de corrélation linéaire ne cachait pas de structure.
- Aucune relation visible dans ce jeu de données ; garder ou écarter la variable se décidera à l'étape de préparation.

## Interactions simples

In [ ]:
# La relation pluie -> rendement est-elle la même pour toutes les cultures ?
utils.use_inline_backend()

tranches_pluie = pd.cut(df["Rainfall_mm"], bins=8)
par_culture = df.pivot_table(
    index=tranches_pluie, columns="Crop",
    values="Yield_tons_per_hectare", aggfunc="mean", observed=True,
)

ax = par_culture.plot(kind="bar", figsize=(13, 5), width=0.85)
ax.set_xlabel("Tranche de pluie (mm)")
ax.set_ylabel("Rendement moyen (t/ha)")
ax.set_title("Rendement moyen par tranche de pluie, culture par culture")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

ecarts = par_culture.max(axis=1) - par_culture.min(axis=1)
print(f"écart maximum entre cultures, à pluie comparable : {ecarts.max():.4f} t/ha")

In [ ]:
# L'effet de l'engrais et de l'irrigation est-il le même pour toutes les cultures ?
for variable in ["Fertilizer_Used", "Irrigation_Used"]:
    table = df.pivot_table(index="Crop", columns=variable, values="Yield_tons_per_hectare")
    table["ecart"] = table[True] - table[False]
    print(f"-- {variable}")
    print(table.round(3).to_string())
    print(f"   variation de l'écart entre cultures : "
          f"{table['ecart'].max() - table['ecart'].min():.4f} t/ha\n")

**Observations :**

- À pluie comparable, les 6 cultures donnent quasiment le même rendement moyen : **0,028 t/ha** d'écart maximum. L'effet de l'engrais varie de +1,494 à +1,514 et celui de l'irrigation de +1,189 à +1,209, soit environ **0,02 t/ha** de variation.
- **Aucune interaction notable dans les vérifications réalisées** : les cultures restent équivalentes à conditions comparables, pas seulement en moyenne.
- Point important pour la suite : `/recommend` doit classer les cultures, or rien ici ne permet de les départager. À confronter au second jeu de données.

## Variables booléennes

In [ ]:
for col in ["Fertilizer_Used", "Irrigation_Used"]:
    print(df[col].value_counts(normalize=True).to_string(), "\n")

pd.crosstab(df["Fertilizer_Used"], df["Irrigation_Used"])

**Observations :**

- Les deux variables sont réparties à **50 / 50**, et leur croisement donne quatre cases d'environ 250 000 lignes.
- Les quatre combinaisons étant également représentées, on pourra mesurer l'effet de l'engrais et celui de l'irrigation séparément.

## Variables Catégorielles

In [ ]:
cat_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]

utils.plot_pies(df, cat_cols)

In [ ]:
# Rendement moyen pour chaque croisement de variables catégorielles
utils.plot_cat_heatmaps(df, "Yield_tons_per_hectare", cat_cols)

**Observations :**

- Les 4 variables texte sont parfaitement équilibrées : 25 % par région, 16,7 % par culture et par type de sol, 33,3 % par météo.
- Aucune différence notable de rendement n'apparaît ici selon la culture, la région, le sol ou la météo : **0,012 t/ha d'écart au maximum** entre modalités, soit 0,2 %.
- Attention à l'échelle des heatmaps : elle se recale sur un intervalle minuscule (4,63 à 4,67) et donne l'illusion de grosses différences ; avec `vmin`/`vmax` fixés, tout devient uniforme.

## Cohérence des données

In [ ]:
# La météo déclarée est-elle liée à la pluie et à la température mesurées ?
print(df.groupby("Weather_Condition")["Rainfall_mm"].agg(["mean", "min", "max"]).round(1), "\n")
print(df.groupby("Weather_Condition")["Temperature_Celsius"].mean().round(2))

**Observations :**

- Pluie moyenne quasi identique selon la météo déclarée : **549,5 / 550,1 / 550,3 mm**, même plage de 100 à 1000 mm, et 27,5 °C dans les trois cas.
- `Weather_Condition` se comporte donc comme si elle avait été tirée **indépendamment** de la pluie et de la température, ce qui est compatible avec une génération synthétique.
- Cela ne suffit pas à dire qu'une ligne « Sunny » très arrosée serait impossible.

# Conclusion

## Ce que contient le jeu de données
1 000 000 de lignes, 10 colonnes, **aucune valeur manquante, aucun doublon**.

## Ce qui explique le rendement
Trois variables dominent nettement :

| Variable | Effet |
|---|---|
| `Rainfall_mm` | le plus fort — corrélation 0,76, relation droite |
| `Fertilizer_Used` | +1,50 t/ha en moyenne |
| `Irrigation_Used` | +1,20 t/ha en moyenne |

Malgré une corrélation faible (0,09), l'analyse par tranches de `Temperature_Celsius` fait
apparaître une tendance régulière : le rendement moyen augmente de **+0,445 t/ha** entre les
températures les plus basses et les plus élevées, sans exception dans la progression. L'effet
est donc **faible relativement à la variabilité totale**, mais **systématique** dans ce jeu de
données.

*Sur les interactions testées, les effets de la pluie, de l'engrais et de l'irrigation sont
très similaires entre les cultures.*

## Ce pour quoi aucune relation n'apparaît
- `Region`, `Soil_Type`, `Crop` et `Weather_Condition` : 0,2 % d'écart au maximum entre
  modalités. Les cultures restent équivalentes à conditions comparables, sur les croisements
  examinés.
- `Days_to_Harvest` : ni relation droite (corrélation 0,00), ni relation en courbe
  (0,029 t/ha d'amplitude entre tranches).
- `Weather_Condition` semble par ailleurs indépendante de la pluie et de la température.


## Les points de qualité à traiter
- **231 rendements négatifs** (0,023 %), tous sans engrais ni irrigation, et 223 sur 231
  avec moins de 200 mm de pluie.
- `Weather_Condition` n'apporte aucune information exploitable en l'état.

## Décisions à confirmer plus tard
- Que faire des 231 rendements négatifs : conserver, ramener à 0, ou supprimer les lignes ?
- Garder ou écarter `Days_to_Harvest`, pour lequel aucune relation n'apparaît ?
- Garder les 4 variables texte, alors qu'aucune ne se relie au rendement ?
- Garder `Temperature_Celsius`, dont l'effet est faible mais régulier ? Les tranches plaident pour la conserver.

## Hypothèses à vérifier plus tard
- Une régression linéaire simple devrait déjà bien marcher, vu que la relation avec la pluie
  est droite et que les effets s'additionnent.
- L'absence d'effet de la culture, y compris à conditions égales, rendra `/recommend` peu
  utile avec ce seul jeu de données.
- Le second jeu de données (CropYield Prediction) permettra de vérifier si la culture a un
  effet ailleurs.
